<a href="https://colab.research.google.com/github/Eva360563/MaskArchitectureAnomaly_CourseProject/blob/main/step5_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Guarda cosa c'è nella cartella training
import os

training_path = "/content/MaskArchitectureAnomaly_CourseProject/eomt/training"
for root, dirs, files in os.walk(training_path):
    for f in files:
        print(os.path.join(root, f).replace(training_path, ""))

In [ ]:
!git clone https://github.com/robertomahamalimage-star/MaskArchitectureAnomaly_CourseProject.git
%cd MaskArchitectureAnomaly_CourseProject

Cloning into 'MaskArchitectureAnomaly_CourseProject'...
remote: Enumerating objects: 131, done.
remote: Total 131 (delta 0), reused 0 (delta 0), pack-reused 131 (from 1)
Receiving objects: 100% (131/131), 26.88 MiB | 32.11 MiB/s, done.
Resolving deltas: 100% (24/24), done.
/content/MaskArchitectureAnomaly_CourseProject


In [ ]:
with open("/content/MaskArchitectureAnomaly_CourseProject/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml") as f:
    print(f.read())

trainer:
  max_epochs: 107
  logger:
    class_path: lightning.pytorch.loggers.wandb.WandbLogger
    init_args:
      resume: allow
      project: "eomt"
      name: "cityscapes_semantic_eomt_base_640"
model:
  class_path: training.mask_classification_semantic.MaskClassificationSemantic
  init_args:
    attn_mask_annealing_enabled: True
    attn_mask_annealing_start_steps: [3317, 8292, 13268]
    attn_mask_annealing_end_steps: [6634, 11609, 16585]
    network:
      class_path: models.eomt.EoMT
      init_args:
        num_q: 100
        num_blocks: 3
        encoder:
          class_path: models.vit.ViT
          init_args:
            backbone_name: vit_base_patch14_reg4_dinov2
data:
  class_path: datasets.cityscapes_semantic.CityscapesSemantic


In [ ]:
with open("/content/MaskArchitectureAnomaly_CourseProject/eomt/main.py") as f:
    print(f.read())

# ---------------------------------------------------------------
# © 2025 Mobile Perception Systems Lab at TU/e. All rights reserved.
# Licensed under the MIT License.
#
# Portions of this file are adapted from PyTorch Lightning,
# used under the Apache 2.0 License.
# ---------------------------------------------------------------


import jsonargparse._typehints as _t
from types import MethodType
from gitignore_parser import parse_gitignore
import logging
import torch
import warnings
from lightning.pytorch import cli
from lightning.pytorch.callbacks import ModelSummary, LearningRateMonitor
from lightning.pytorch.loops.training_epoch_loop import _TrainingEpochLoop
from lightning.pytorch.loops.fetchers import _DataFetcher, _DataLoaderIterDataFetcher

from training.lightning_module import LightningModule
from datasets.lightning_data_module import LightningDataModule

# Suppress PyTorch FX warnings for DINOv3 models
import os
os.environ["TORCH_LOGS"] = "-dynamo"


_orig_single = _t.raise

In [ ]:
!pip install torch torchvision timm
!pip install lightning -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 40.2 MB/s eta 0:00:00


In [ ]:
import os, sys, yaml, warnings, importlib
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.nn import functional as F
from torch.amp.autocast_mode import autocast
from lightning import seed_everything

seed_everything(0, verbose=False)

# Paths - adatta se necessario
REPO = "/content/MaskArchitectureAnomaly_CourseProject/eomt"
CITYSCAPES_DATA = "/content/drive/MyDrive/Trained_Datasets"
CKPT_CS   = "/content/drive/MyDrive/Trained_Datasets/Copia di eomt_cityscapes.bin"
CKPT_COCO = "/content/drive/MyDrive/Trained_Datasets/Copia di eomt_coco.bin"
CKPT_FINETUNED = "/content/drive/MyDrive/eomt_finetuned_phase1.bin"
CKPT_FINETUNED2 = "/content/drive/MyDrive/eomt_finetuned.bin"
DEVICE = 0  # GPU

sys.path.insert(0, REPO)
os.chdir(REPO)
print("Setup ok")

Setup ok


In [ ]:
def load_model(config_path, ckpt_path, data_path):
    with open(config_path) as f:
        config = yaml.safe_load(f)

    # Carica dataset
    data_module_name, class_name = config["data"]["class_path"].rsplit(".", 1)
    data_cls = getattr(importlib.import_module(data_module_name), class_name)
    data_kwargs = config["data"].get("init_args", {})
    data = data_cls(
        path=data_path, batch_size=1, num_workers=0,
        check_empty_targets=False, **data_kwargs
    ).setup()

    # Carica encoder
    enc_cfg = config["model"]["init_args"]["network"]["init_args"]["encoder"]
    enc_cls = getattr(importlib.import_module(*enc_cfg["class_path"].rsplit(".",1)[::-1].__reversed__().__next__() and enc_cfg["class_path"].rsplit(".",1)), enc_cfg["class_path"].rsplit(".",1)[1])

    # (versione più leggibile)
    enc_module, enc_classname = enc_cfg["class_path"].rsplit(".", 1)
    enc_cls = getattr(importlib.import_module(enc_module), enc_classname)
    encoder = enc_cls(img_size=data.img_size, **enc_cfg.get("init_args", {}))

    # Carica network
    net_cfg = config["model"]["init_args"]["network"]
    net_module, net_classname = net_cfg["class_path"].rsplit(".", 1)
    net_cls = getattr(importlib.import_module(net_module), net_classname)
    net_kwargs = {k: v for k, v in net_cfg["init_args"].items() if k != "encoder"}
    network = net_cls(masked_attn_enabled=False, num_classes=data.num_classes,
                      encoder=encoder, **net_kwargs)

    # Carica Lightning module
    lit_module, lit_classname = config["model"]["class_path"].rsplit(".", 1)
    lit_cls = getattr(importlib.import_module(lit_module), lit_classname)
    model_kwargs = {k: v for k, v in config["model"]["init_args"].items() if k != "network"}
    if "stuff_classes" in config["data"].get("init_args", {}):
        model_kwargs["stuff_classes"] = config["data"]["init_args"]["stuff_classes"]

    model = lit_cls(img_size=data.img_size, num_classes=data.num_classes,
                    network=network, **model_kwargs).eval().to(DEVICE)

    # Carica pesi LOCALI (dal Drive)
    state_dict = torch.load(ckpt_path, map_location=f"cuda:{DEVICE}", weights_only=True)
    model_state = model.state_dict()

    filtered = {}
    skipped = []
    for k, v in state_dict.items():
        if k in model_state and model_state[k].shape == v.shape:
            filtered[k] = v
        else:
            skipped.append(f"  SKIP {k}: checkpoint {list(v.shape)} vs model {list(model_state[k].shape) if k in model_state else 'non esiste'}")

    model.load_state_dict(filtered, strict=False)
    print(f"✅ Caricati {len(filtered)}/{len(state_dict)} layer da {ckpt_path}")
    if skipped:
        print(f"⚠️  Layer saltati ({len(skipped)}):")
        for s in skipped: print(s)

    return model, data

In [ ]:
model_cs, data_cs = load_model(
    config_path="/content/MaskArchitectureAnomaly_CourseProject/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml",
    ckpt_path=CKPT_CS,
    data_path=CITYSCAPES_DATA
)

✅ Caricati 198/198 layer da /content/drive/MyDrive/Trained_Datasets/Copia di eomt_cityscapes.bin


In [ ]:
# Riusa model_cs che hai già caricato
print("Layer del modello EoMT:")
for name, param in model_cs.named_parameters():
    print(f"  {name:80s} | shape: {list(param.shape)}")

Layer del modello EoMT:
  network.encoder.backbone.cls_token                                               | shape: [1, 1, 768]
  network.encoder.backbone.reg_token                                               | shape: [1, 4, 768]
  network.encoder.backbone.pos_embed                                               | shape: [1, 4096, 768]
  network.encoder.backbone.patch_embed.proj.weight                                 | shape: [768, 3, 16, 16]
  network.encoder.backbone.patch_embed.proj.bias                                   | shape: [768]
  network.encoder.backbone.blocks.0.norm1.weight                                   | shape: [768]
  network.encoder.backbone.blocks.0.norm1.bias                                     | shape: [768]
  network.encoder.backbone.blocks.0.attn.qkv.weight                                | shape: [2304, 768]
  network.encoder.backbone.blocks.0.attn.qkv.bias                                  | shape: [2304]
  network.encoder.backbone.blocks.0.attn.proj.weight   

In [ ]:
# Carica architettura Cityscapes (19 classi) ma con pesi COCO
model_ft, data_ft = load_model(
    config_path="/content/MaskArchitectureAnomaly_CourseProject/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml",
    ckpt_path=CKPT_COCO,   # ← pesi COCO, non Cityscapes!
    data_path=CITYSCAPES_DATA
)
# strict=False nella load_model fa sì che class_head (dimensione diversa)
# parta da zero, mentre il backbone carica i pesi COCO
model_ft.train()
print("✅ Modello pronto per fine-tuning")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.


✅ Caricati 193/198 layer da /content/drive/MyDrive/Trained_Datasets/Copia di eomt_coco.bin
⚠️  Layer saltati (5):
  SKIP network.encoder.backbone.pos_embed: checkpoint [1, 1600, 768] vs model [1, 4096, 768]
  SKIP network.q.weight: checkpoint [200, 768] vs model [100, 768]
  SKIP network.class_head.weight: checkpoint [134, 768] vs model [20, 768]
  SKIP network.class_head.bias: checkpoint [134] vs model [20]
  SKIP criterion.empty_weight: checkpoint [134] vs model [20]
✅ Modello pronto per fine-tuning


In [ ]:
# Congela tutto il backbone DINOv2
for name, param in model_ft.named_parameters():
    if "network.encoder.backbone" in name:
        param.requires_grad = False
    else:
        param.requires_grad = True

# Controlla quanti parametri sono trainabili
total = sum(p.numel() for p in model_ft.parameters())
trainable = sum(p.numel() for p in model_ft.parameters() if p.requires_grad)
print(f"Parametri totali:     {total:,}")
print(f"Parametri trainabili: {trainable:,} ({100*trainable/total:.1f}%)")

Parametri totali:     95,415,572
Parametri trainabili: 6,600,980 (6.9%)


In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import GradScaler, autocast

EPOCHS_PHASE1 = 5
EPOCHS_PHASE2 = 5
LR = 1e-4
SAVE_PATH = "/content/drive/MyDrive/eomt_finetuned.bin"

optimizer = AdamW(
    [p for p in model_ft.parameters() if p.requires_grad],
    lr=LR, weight_decay=1e-4
)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS_PHASE1)
scaler = GradScaler()

train_loader = data_ft.train_dataloader()
print(f"✅ Training loader: {len(train_loader)} batch per epoca")

✅ Training loader: 2975 batch per epoca


In [ ]:
# Cella debug - esegui prima di riprovare il training
batch = next(iter(train_loader))
print("Tipo batch:", type(batch))
print("Lunghezza batch:", len(batch))
for i, b in enumerate(batch):
    print(f"  batch[{i}]: tipo={type(b)}, ", end="")
    if isinstance(b, torch.Tensor):
        print(f"shape={b.shape}")
    elif isinstance(b, list):
        print(f"lista di {len(b)} elementi, primo elemento: tipo={type(b[0])}", end="")
        if isinstance(b[0], torch.Tensor): print(f", shape={b[0].shape}")
        elif isinstance(b[0], dict): print(f", chiavi={list(b[0].keys())}")
        else: print()
    elif isinstance(b, dict):
        print(f"dict con chiavi={list(b.keys())}")
    else:
        print()

Tipo batch: <class 'tuple'>
Lunghezza batch: 2
  batch[0]: tipo=<class 'torch.Tensor'>, shape=torch.Size([1, 3, 1024, 1024])
  batch[1]: tipo=<class 'list'>, lista di 1 elementi, primo elemento: tipo=<class 'dict'>, chiavi=['masks', 'labels', 'is_crowd']


In [ ]:
import time

def move_to_device(batch, device):
    imgs = batch[0].to(device)  # [B, 3, H, W]
    targets = []
    for t in batch[1]:  # lista di dict
        targets.append({
            k: v.to(device) if isinstance(v, torch.Tensor) else v
            for k, v in t.items()
        })
    return imgs, targets

def train_one_epoch(model, loader, optimizer, scaler, epoch):
    model.train()
    total_loss = 0
    start = time.time()

    for i, batch in enumerate(loader):
        imgs, targets = move_to_device(batch, DEVICE)

        optimizer.zero_grad()

        with autocast(device_type="cuda", dtype=torch.float16):
            # Passa il batch nel formato corretto
            loss_dict = model.training_step((imgs, targets), i)
            if isinstance(loss_dict, torch.Tensor):
                loss = loss_dict
            else:
                loss = sum(loss_dict.values())

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.01)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

        if i % 200 == 0:
            elapsed = time.time() - start
            print(f"  Epoch {epoch} | Batch {i}/{len(loader)} | Loss: {loss.item():.4f} | {elapsed:.0f}s")

    return total_loss / len(loader)

print("🚀 Fase 1: addestro solo la prediction head...")
for epoch in range(1, EPOCHS_PHASE1 + 1):
    avg_loss = train_one_epoch(model_ft, train_loader, optimizer, scaler, epoch)
    scheduler.step()
    print(f"✅ Epoch {epoch}/{EPOCHS_PHASE1} completata | Loss media: {avg_loss:.4f}")

torch.save(model_ft.state_dict(), SAVE_PATH.replace(".bin", "_phase1.bin"))
print("💾 Checkpoint Fase 1 salvato!")

🚀 Fase 1: addestro solo la prediction head...


/usr/local/lib/python3.12/dist-packages/lightning/pytorch/core/module.py:451: You are trying to `self.log()` but the `self.trainer` reference is not registered on the model yet. This is most likely because the model hasn't been passed to the `Trainer`


  Epoch 1 | Batch 0/2975 | Loss: 13.2375 | 5s
  Epoch 1 | Batch 200/2975 | Loss: 6.5038 | 290s
  Epoch 1 | Batch 400/2975 | Loss: 5.7683 | 464s
  Epoch 1 | Batch 600/2975 | Loss: 2.9974 | 629s
  Epoch 1 | Batch 800/2975 | Loss: 3.3838 | 793s
  Epoch 1 | Batch 1000/2975 | Loss: 2.0693 | 956s
  Epoch 1 | Batch 1200/2975 | Loss: 3.2736 | 1116s
  Epoch 1 | Batch 1400/2975 | Loss: 2.7036 | 1276s
  Epoch 1 | Batch 1600/2975 | Loss: 3.7018 | 1443s
  Epoch 1 | Batch 1800/2975 | Loss: 2.3786 | 1605s
  Epoch 1 | Batch 2000/2975 | Loss: 1.4917 | 1769s
  Epoch 1 | Batch 2200/2975 | Loss: 2.5333 | 1935s
  Epoch 1 | Batch 2400/2975 | Loss: 1.6085 | 2099s
  Epoch 1 | Batch 2600/2975 | Loss: 2.0311 | 2267s
  Epoch 1 | Batch 2800/2975 | Loss: 2.5624 | 2432s
✅ Epoch 1/5 completata | Loss media: 3.0495
  Epoch 2 | Batch 0/2975 | Loss: 1.4944 | 1s
  Epoch 2 | Batch 200/2975 | Loss: 1.1551 | 165s
  Epoch 2 | Batch 400/2975 | Loss: 3.2026 | 327s
  Epoch 2 | Batch 600/2975 | Loss: 3.1898 | 484s
  Epoch 2 | B

In [ ]:
import time
from torch.amp import GradScaler, autocast

scaler = GradScaler()

def move_to_device(batch, device):
    imgs = batch[0].to(device)
    targets = []
    for t in batch[1]:
        targets.append({
            k: v.to(device) if isinstance(v, torch.Tensor) else v
            for k, v in t.items()
        })
    return imgs, targets

def train_one_epoch(model, loader, optimizer, scaler, epoch):
    model.train()
    total_loss = 0
    start = time.time()

    for i, batch in enumerate(loader):
        imgs, targets = move_to_device(batch, DEVICE)

        optimizer.zero_grad()

        with autocast(device_type="cuda", dtype=torch.float16):
            loss_dict = model.training_step((imgs, targets), i)
            if isinstance(loss_dict, torch.Tensor):
                loss = loss_dict
            else:
                loss = sum(loss_dict.values())

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.01)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

        if i % 200 == 0:
            elapsed = time.time() - start
            print(f"  Epoch {epoch} | Batch {i}/{len(loader)} | Loss: {loss.item():.4f} | {elapsed:.0f}s")

    return total_loss / len(loader)

In [ ]:
# Sblocca blocks 9, 10, 11 e la norm finale
for name, param in model_ft.named_parameters():
    if any(f"backbone.blocks.{i}" in name for i in [9, 10, 11]):
        param.requires_grad = True
    if "backbone.norm" in name:
        param.requires_grad = True

trainable = sum(p.numel() for p in model_ft.parameters() if p.requires_grad)
print(f"Parametri trainabili Fase 2: {trainable:,}")

optimizer2 = AdamW(
    [p for p in model_ft.parameters() if p.requires_grad],
    lr=LR / 10,  # lr più basso per i layer profondi
    weight_decay=1e-4
)
scheduler2 = CosineAnnealingLR(optimizer2, T_max=EPOCHS_PHASE2)

print("🚀 Fase 2: addestro head + ultimi 3 blocchi...")
for epoch in range(1, EPOCHS_PHASE2 + 1):
    avg_loss = train_one_epoch(model_ft, train_loader, optimizer2, scaler, epoch)
    scheduler2.step()
    # Salva dopo ogni epoca
    torch.save(model_ft.state_dict(),
               SAVE_PATH.replace(".bin", f"_phase2_epoch{epoch}.bin"))
    print(f"✅ Epoch {epoch}/{EPOCHS_PHASE2} | Loss: {avg_loss:.4f} | checkpoint salvato")

# Salva checkpoint finale
torch.save(model_ft.state_dict(), SAVE_PATH)
print(f"💾 Checkpoint finale salvato in {SAVE_PATH}")

Parametri trainabili Fase 2: 27,870,740
🚀 Fase 2: addestro head + ultimi 3 blocchi...


/usr/local/lib/python3.12/dist-packages/lightning/pytorch/core/module.py:451: You are trying to `self.log()` but the `self.trainer` reference is not registered on the model yet. This is most likely because the model hasn't been passed to the `Trainer`


  Epoch 1 | Batch 0/2975 | Loss: 12.3908 | 10s
  Epoch 1 | Batch 200/2975 | Loss: 5.3600 | 407s
  Epoch 1 | Batch 400/2975 | Loss: 3.9333 | 585s
  Epoch 1 | Batch 600/2975 | Loss: 2.4400 | 756s
  Epoch 1 | Batch 800/2975 | Loss: 3.5416 | 925s
  Epoch 1 | Batch 1000/2975 | Loss: 2.7607 | 1100s
  Epoch 1 | Batch 1200/2975 | Loss: 2.9680 | 1273s
  Epoch 1 | Batch 1400/2975 | Loss: 2.1019 | 1442s
  Epoch 1 | Batch 1600/2975 | Loss: 2.9563 | 1612s
  Epoch 1 | Batch 1800/2975 | Loss: 2.7528 | 1784s
  Epoch 1 | Batch 2000/2975 | Loss: 3.2629 | 1949s
  Epoch 1 | Batch 2200/2975 | Loss: 2.3216 | 2120s
  Epoch 1 | Batch 2400/2975 | Loss: 2.4154 | 2287s
  Epoch 1 | Batch 2600/2975 | Loss: 2.0743 | 2459s
  Epoch 1 | Batch 2800/2975 | Loss: 1.7710 | 2631s
✅ Epoch 1/5 | Loss: 2.8082 | checkpoint salvato
  Epoch 2 | Batch 0/2975 | Loss: 1.4500 | 1s
  Epoch 2 | Batch 200/2975 | Loss: 2.1208 | 176s
  Epoch 2 | Batch 400/2975 | Loss: 1.4684 | 350s
  Epoch 2 | Batch 600/2975 | Loss: 2.0241 | 519s
  Epoch

In [ ]:
def valuta_modello(model, data, nome):
    from torchmetrics.classification import MulticlassJaccardIndex
    model.eval()
    miou = MulticlassJaccardIndex(num_classes=19, ignore_index=255).to(DEVICE)

    print(f"Valuto {nome}...")
    for i, batch in enumerate(data.val_dataloader()):
        imgs = batch[0]
        targets = batch[1]
        if isinstance(imgs, (list, tuple)): img = imgs[0]
        else: img = imgs.squeeze(0)
        if isinstance(targets, (list, tuple)): target = targets[0]
        else: target = targets.squeeze(0)

        with torch.no_grad(), autocast(device_type="cuda", dtype=torch.float16):
            imgs_input = [img.to(DEVICE)]
            img_sizes = [img.shape[-2:]]
            crops, origins = model.window_imgs_semantic(imgs_input)
            mask_logits_per_layer, class_logits_per_layer = model(crops)
            mask_logits = F.interpolate(mask_logits_per_layer[-1], data.img_size, mode="bilinear")
            crop_logits = model.to_per_pixel_logits_semantic(mask_logits, class_logits_per_layer[-1])
            logits = model.revert_window_logits_semantic(crop_logits, origins, img_sizes)
            pred = logits[0].argmax(0)

        gt = model.to_per_pixel_targets_semantic([target], 255)[0].to(DEVICE)
        miou.update(pred.unsqueeze(0), gt.unsqueeze(0))

        if i % 50 == 0:
            print(f"  {i}/500...")

    result = miou.compute()
    print(f"✅ {nome}: {result.item()*100:.2f}%\n")
    return result.item()*100

In [ ]:
model_phase1, data_ft = load_model(
    config_path="configs/dinov2/cityscapes/semantic/eomt_base_640.yaml",
    ckpt_path=CKPT_FINETUNED,
    data_path=CITYSCAPES_DATA
)
miou_phase1 = valuta_modello(model_phase1, data_ft, "EoMT fine-tuned Fase 1")

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.


✅ Caricati 198/198 layer da /content/drive/MyDrive/eomt_finetuned_phase1.bin
Valuto EoMT fine-tuned Fase 1...
  0/500...
  50/500...
  100/500...
  150/500...
  200/500...
  250/500...
  300/500...
  350/500...
  400/500...
  450/500...
✅ EoMT fine-tuned Fase 1: 68.82%



In [ ]:
model_phase2, data_ft = load_model(
    config_path="configs/dinov2/cityscapes/semantic/eomt_base_640.yaml",
    ckpt_path=CKPT_FINETUNED2,
    data_path=CITYSCAPES_DATA
)
miou_phase2 = valuta_modello(model_phase2, data_ft, "EoMT fine-tuned Fase 2")

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.


✅ Caricati 198/198 layer da /content/drive/MyDrive/eomt_finetuned.bin
Valuto EoMT fine-tuned Fase 2...
  0/500...
  50/500...
  100/500...
  150/500...
  200/500...
  250/500...
  300/500...
  350/500...
  400/500...
  450/500...
✅ EoMT fine-tuned Fase 2: 76.42%



In [ ]:
print("=" * 50)
print("📊 RISULTATI FINALI STEP 5")
print("=" * 50)
print(f"  EoMT-COCO originale      : 25.11%")
print(f"  EoMT-CS originale        : 81.68%")
print(f"  EoMT fine-tuned Fase 1   : {miou_phase1:.2f}%")
print(f"  EoMT fine-tuned Fase 2   : {miou_phase2:.2f}%")
print("=" * 50)

with open("/content/drive/MyDrive/risultati_step5.txt", "w") as f:
    f.write(f"EoMT-COCO originale: 25.11%\n")
    f.write(f"EoMT-CS originale: 81.68%\n")
    f.write(f"EoMT fine-tuned Fase1: {miou_phase1:.2f}%\n")
    f.write(f"EoMT fine-tuned Fase2: {miou_phase2:.2f}%\n")
print("✅ Risultati salvati su Drive!")

📊 RISULTATI FINALI STEP 5
  EoMT-COCO originale      : 25.11%
  EoMT-CS originale        : 81.68%
  EoMT fine-tuned Fase 1   : 68.82%
  EoMT fine-tuned Fase 2   : 76.42%
✅ Risultati salvati su Drive!


In [ ]:
import os

drive_root = "/content/drive/MyDrive"
print("Checkpoint trovati:")
for f in os.listdir(drive_root):
    if "eomt_finetuned" in f or "eomt" in f.lower():
        size = os.path.getsize(os.path.join(drive_root, f)) / (1024*1024)
        print(f"  {f}  ({size:.0f} MB)")

Checkpoint trovati:
  eomt_finetuned.bin  (364 MB)
  eomt_finetuned_phase1.bin  (364 MB)
  eomt_finetuned_phase2  (0 MB)


In [ ]:
from torchmetrics.classification import MulticlassJaccardIndex

# Nomi delle 19 classi di Cityscapes in ordine
CITYSCAPES_CLASSES = [
    "road", "sidewalk", "building", "wall", "fence",
    "pole", "traffic light", "traffic sign", "vegetation", "terrain",
    "sky", "person", "rider", "car", "truck",
    "bus", "train", "motorcycle", "bicycle"
]

def valuta_per_classe(model, data, nome):
    # average=None → restituisce IoU per ogni classe separatamente
    miou = MulticlassJaccardIndex(
        num_classes=19, ignore_index=255, average=None
    ).to(DEVICE)
    model.eval()

    print(f"Valuto {nome}...")
    for i, batch in enumerate(data.val_dataloader()):
        imgs = batch[0]
        targets = batch[1]
        if isinstance(imgs, (list, tuple)): img = imgs[0]
        else: img = imgs.squeeze(0)
        if isinstance(targets, (list, tuple)): target = targets[0]
        else: target = targets.squeeze(0)

        with torch.no_grad(), autocast(device_type="cuda", dtype=torch.float16):
            imgs_input = [img.to(DEVICE)]
            img_sizes = [img.shape[-2:]]
            crops, origins = model.window_imgs_semantic(imgs_input)
            mask_logits_per_layer, class_logits_per_layer = model(crops)
            mask_logits = F.interpolate(mask_logits_per_layer[-1], data.img_size, mode="bilinear")
            crop_logits = model.to_per_pixel_logits_semantic(mask_logits, class_logits_per_layer[-1])
            logits = model.revert_window_logits_semantic(crop_logits, origins, img_sizes)
            pred = logits[0].argmax(0)

        gt = model.to_per_pixel_targets_semantic([target], 255)[0].to(DEVICE)
        miou.update(pred.unsqueeze(0), gt.unsqueeze(0))

        if i % 50 == 0:
            print(f"  {i}/500...")

    per_class = miou.compute()
    return per_class

# Valuta entrambi i modelli
iou_cs     = valuta_per_classe(model_cs,     data_ft, "EoMT-CS originale")
iou_ft     = valuta_per_classe(model_phase2, data_ft, "EoMT fine-tuned")

# Stampa tabella
print(f"\n{'Classe':<20} {'EoMT-CS originale':>20} {'EoMT fine-tuned':>18}")
print("-" * 60)
for i, cls in enumerate(CITYSCAPES_CLASSES):
    print(f"{cls:<20} {iou_cs[i].item()*100:>19.2f}% {iou_ft[i].item()*100:>17.2f}%")
print("-" * 60)
print(f"{'mIoU medio':<20} {iou_cs.mean().item()*100:>19.2f}% {iou_ft.mean().item()*100:>17.2f}%")

# Salva su Drive
with open("/content/drive/MyDrive/risultati_step5_perclasse.txt", "w") as f:
    f.write(f"{'Classe':<20} {'EoMT-CS originale':>20} {'EoMT fine-tuned':>18}\n")
    f.write("-" * 60 + "\n")
    for i, cls in enumerate(CITYSCAPES_CLASSES):
        f.write(f"{cls:<20} {iou_cs[i].item()*100:>19.2f}% {iou_ft[i].item()*100:>17.2f}%\n")
    f.write("-" * 60 + "\n")
    f.write(f"{'mIoU medio':<20} {iou_cs.mean().item()*100:>19.2f}% {iou_ft.mean().item()*100:>17.2f}%\n")
print("✅ Tabella salvata su Drive!")

Valuto EoMT-CS originale...
  0/500...
  50/500...
  100/500...
  150/500...
  200/500...
  250/500...
  300/500...
  350/500...
  400/500...
  450/500...
Valuto EoMT fine-tuned...
  0/500...
  50/500...
  100/500...
  150/500...
  200/500...
  250/500...
  300/500...
  350/500...
  400/500...
  450/500...

Classe                  EoMT-CS originale    EoMT fine-tuned
------------------------------------------------------------
road                               98.40%             97.78%
sidewalk                           87.36%             82.80%
building                           94.15%             92.51%
wall                               66.07%             59.55%
fence                              65.49%             56.58%
pole                               71.04%             58.02%
traffic light                      75.00%             67.55%
traffic sign                       82.13%             75.44%
vegetation                         93.02%             91.92%
terrain             